In [ ]:
## Project Architecture:
## Customer Reviews Data --> Load using Pandas --> Text Chunking --> Creating Embedding --> Store that in FAISS --> Retrieve the Data --> Agument with Gemini LLM --> Create Conversational Bot

In [ ]:
!pip install faiss-cpu

In [ ]:
import pandas as pd
import numpy as np
from google import genai
import faiss


In [ ]:
df = pd.read_csv("/content/customer_reviews.csv")
df

,review_id,product_id,review_text,rating
0,1,P101,Excellent battery life and superb camera quali...,5
1,2,P101,"The phone lasts two days on a single charge, g...",5
2,3,P101,Battery is good but the screen is average.,4
3,4,P101,Amazing camera but the design feels outdated.,4
4,5,P101,Very reliable battery performance.,5
5,6,P102,"Poor performance, lags frequently when opening...",2
6,7,P102,"Camera quality is below average, not recommended.",2
7,8,P102,The phone heats up quickly during use.,1
8,9,P102,Touch response is slow and frustrating.,2
9,10,P102,Good design but very slow.,2


In [ ]:
df.columns

Index(['review_id', 'product_id', 'review_text', 'rating'], dtype='object')

In [ ]:
documents = []
for index, row in df.iterrows():
  text = f"""
  Product: {row["product_id"]},
  Rating: {row["rating"]},
  Review: {row["review_text"]}
  """
  documents.append(text)

print(documents[0]) ## fetching 1st document details from customer_review data


  Product: P101,
  Rating: 5,
  Review: Excellent battery life and superb camera quality. Highly recommended for travelers.
  


In [ ]:
print(len(documents))

50


In [ ]:
## Creating Embedding (to store it in vector DB to perform semantic search)

from sentence_transformers import SentenceTransformer
model = SentenceTransformer("all-MiniLM-L6-v2")
print(model)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({})
)


In [ ]:
embeddings = model.encode(documents)
print(embeddings)

[[-0.06270523  0.00371411 -0.05552196 ... -0.00159856 -0.00976798
   0.09007781]
 [-0.06452829  0.05130015  0.03992397 ...  0.00206818 -0.02030024
   0.08154477]
 [-0.06796391  0.0180227  -0.03917669 ...  0.00432145  0.00212555
   0.08064228]
 ...
 [-0.04195224  0.02343745 -0.06947212 ... -0.02306841 -0.08461837
   0.08456996]
 [-0.04453544  0.03503567 -0.03655394 ... -0.085401   -0.05100541
   0.06451549]
 [-0.06631995  0.04537386 -0.04105097 ... -0.08083738 -0.08307622
   0.08726691]]


In [ ]:
## Store that in Vector Database - FAISS

dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))

In [ ]:
from decorator import append
### Creating the Retrieval Function to Retrieve Data from vector DB, which will act as a User context for LLM

def retrieve_context(query,top_k=3):
  query_embedding = model.encode([query])
  distances, indices = index.search(np.array(query_embedding), k = top_k)

  retrieved_docs =[]
  for idx in indices[0]:
    retrieved_docs.append(documents[idx])

  return "\n\n".join(retrieved_docs)


In [ ]:
## Conversational Bot (Agumentation - RAG based Chatbot)
from google.colab import userdata
apk = userdata.get('MY_API_KEY3')
client = genai.Client(api_key = apk)

In [ ]:
chat_history = []
def ask_question(question):
  global chat_history
  context = retrieve_context(question,top_k=3)
  history = "\n".join(chat_history)

  prompt= f"""
  You are a Customer Review Analyst Assistant.

  Conversation Histor:
  {history}
  Context:
  {context}
  Question:
  {question}

  Instruction:
  - Answer from the reviews only
  - Summerize customer opinions
  - Keep answers conversational
  - Mention common issues if any
  """

  response = client.models.generate_content(
      model = "gemini-2.5-flash",
      contents = prompt
  )
  answer = response.text
  chat_history.append(f"Question: ,{question}")
  chat_history.append(f"Answer: ,{answer}")

  return answer

In [ ]:
response = ask_question(
    "Which product is good for long battery?"
)
print(response)

Based on the customer reviews we have here:

It looks like **Product P109** is generally highlighted as being good for long battery life. One customer specifically praised its "long battery backup."

However, it's worth noting a common issue brought up for P109: while it charges quickly, another customer mentioned that the "battery drains in heavy use." So, if you're a heavy user, that might be something to consider.

On the other hand, **Product P103** received feedback that its "battery drains fast," so it might not be the ideal choice if you're looking for extended battery life.


In [ ]:
response = ask_question(
    "What are the most common complaints?"
)
print(response)

Based on these reviews, here are the most common complaints:

*   For **Product P102**, a significant complaint is that the "Touch response is slow and frustrating."
*   **Product P104** users mentioned that while it's lightweight, it "gets scratched easily."
*   And with **Product P107**, the main issue highlighted is that its "edges chip over time," despite feeling strong initially.

So, it seems like each product has its own distinct issue, ranging from slow touch response to durability concerns like getting scratched or chipped.


### Creating pickel file for streamlit application

In [ ]:
import pickle

In [ ]:
with open("faiss_index.pkl","wb") as f:
  pickle.dump(index,f)
with open("documents.pkl","wb") as f:
  pickle.dump(documents,f)
print("faiss_index.pkl and documents.pkl saved!")

faiss_index.pkl and documents.pkl saved!
